# Computational Tools for Climate Science - Project 2026
---
**Group 2 - Chaac**

Authors: *Abiyo Gladys, Alex Blackmer, Caroline Roper, Han Nguyen, Sheila Aguilar Palpa*

Date: *July 13-24th*

---

This project explores the relationship between precipitation and crop production in Southeast Asia. Specifically, we tie the Standardized Precipitation Index (SPI) derived from Climate Hazards Group InfraRed Precipitation with Station data (CHIRPS) with cereal production yield anomalies sourced from the Food and Agriculture Organization (FAO). This study considers two domains in Southeast Asia; the mainland (Monsoon Climate Region - MCR) and maritime (Equatorial Climate Region - ECR), focusing on the countries of Vietnam and Indonesia respectively.

In [11]:
# Install packages not included in base environment
!pip install cartopy
!pip install gdown
!pip install xclim

In [12]:
# Imports
import os
# Scientific data packages
import numpy as np
import xarray as xr
import pandas as pd
import datetime
# Plotting libraries
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
# Climate index package
import xclim

---
# Data Management
## Mount Google Drive
First step is to mount the Google Drive fonder containing group project data

In [13]:
# Mount Google Drive project data path
from google.colab import drive
drive.mount('/content/drive')
path_gd =  "/content/drive/MyDrive/Data/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Downloading data
This is a helper function for downloading data

---

### Download CHIRPS data

---
## Load Datasets
This section loads the country masks, precipitation, and cereal production data used for analysis

### Precipitation

In [14]:
path_gd = '/content/drive/MyDrive/Group Project Folder/Data/'

In [15]:
precip_ds = xr.open_dataset(path_gd + "chirps_sel.nc")
precip_ds

<xarray.Dataset> Size: 2GB
Dimensions:    (time: 546, latitude: 900, longitude: 1200)
Coordinates:
  * time       (time) datetime64[ns] 4kB 1981-01-01 1981-02-01 ... 2026-06-01
  * latitude   (latitude) float32 4kB -19.98 -19.92 -19.88 ... 24.88 24.92 24.97
  * longitude  (longitude) float32 5kB 90.02 90.08 90.12 ... 149.9 149.9 150.0
Data variables:
    precip     (time, latitude, longitude) float32 2GB ...
Attributes: (12/14)
    Conventions:       CF-1.6
    title:             CHIRPS Version 3.0
    history:           created by Climate Hazards Center
    version:           Version 3.0
    data_created:      2026-07-14
    creator_name:      Pete Peterson
    ...                ...
    documentation:     http://pubs.usgs.gov/ds/832/
    reference:         Funk, C.C., Peterson, P.J., Landsfeld, M.F., Pedreros,...
    acknowledgements:  The Climate Hazards Center InfraRed Precipitation with...
    ftp_url:           ftp://chg-ftpout.geog.ucsb.edu/pub/chg/products/CHIRPS...
    website:           http://chg.geog.ucsb.edu/data/chirps/index.html
    faq:               http://chg-wiki.geog.ucsb.edu/wiki/CHIRPS_FAQ

### Country Mask

In [16]:
country_mask = xr.open_dataset(path_gd + 'CountryMask/NetCDF/CountryMergedNC.nc')
country_mask = country_mask.rename({"lat": "latitude", "lon": "longitude"})

country_mask = country_mask["country_mask"].interp(
    latitude=precip_ds.latitude,
    longitude=precip_ds.longitude,
    method="nearest"
)

### Cereal Production

In [17]:
cereal_df = pd.read_csv(path_gd + "FAOSTAT_data_en_7-18-2026.csv")

---
## Subset Precipitation Data By Country

In [18]:
# Mask each contry area and drop the NaNs
precip_ind = precip_ds.precip.where(country_mask == 1, drop=True)
precip_viet = precip_ds.precip.where(country_mask == 2, drop=True)


In [19]:
precip_viet_1d = precip_viet.mean(dim=["latitude", "longitude"])
spi_1m_viet = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_viet_1d, freq='MS', window=1)
spi_6m_viet = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_viet_1d, freq='MS', window=6)
spi_12m_viet = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_viet_1d, freq='MS', window=12)

/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:77: UserWarning: Variable does not have a `cell_methods` attribute.
  _check_cell_methods(getattr(vardata, "cell_methods", None), data["cell_methods"])
/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:79: UserWarning: Variable has a non-conforming standard_name: Got `convective precipitation rate`, expected `['precipitation_flux']`
  check_valid(vardata, "standard_name", data["standard_name"])
/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:77: UserWarning: Variable does not have a `cell_methods` attribute.
  _check_cell_methods(getattr(vardata, "cell_methods", None), data["cell_methods"])
/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:79: UserWarning: Variable has a non-conforming standard_name: Got `convective precipitation rate`, expected `['precipitation_flux']`
  check_valid(vardata, "standard_name", data["standard_name"])
/usr/local/lib/python3.12/dist-packages/xclim/core/c

In [20]:
precip_indonesia = precip_ds.precip.where(country_mask == 1, drop=True)

In [21]:
precip_indonesia_1d = precip_indonesia.mean(dim=["latitude", "longitude"])

In [22]:
spi_1m_indonesia = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_indonesia_1d, freq='MS', window=1)
spi_12m_indonesia = xclim.indicators.atmos.standardized_precipitation_index(pr=precip_indonesia_1d, freq='MS', window=12)

/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:77: UserWarning: Variable does not have a `cell_methods` attribute.
  _check_cell_methods(getattr(vardata, "cell_methods", None), data["cell_methods"])
/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:79: UserWarning: Variable has a non-conforming standard_name: Got `convective precipitation rate`, expected `['precipitation_flux']`
  check_valid(vardata, "standard_name", data["standard_name"])
/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:77: UserWarning: Variable does not have a `cell_methods` attribute.
  _check_cell_methods(getattr(vardata, "cell_methods", None), data["cell_methods"])
/usr/local/lib/python3.12/dist-packages/xclim/core/cfchecks.py:79: UserWarning: Variable has a non-conforming standard_name: Got `convective precipitation rate`, expected `['precipitation_flux']`
  check_valid(vardata, "standard_name", data["standard_name"])


In [23]:
### Annual SPI Metrics

In [24]:
def monthly_severity_per_year(x):
  '''takes x, an xarray.DataArray with variable 'spi' monthly and outputs a pandas dataframe of the number of months at each severity level per year'''
  yearly = x.groupby('time.year').map(lambda x: x.groupby_bins(x, bins=[-np.inf, -2, -1.5, -1, 0, np.inf]).count()).fillna(0)
  yearly = yearly.to_dataframe().pivot_table(index = 'year', columns = 'spi_bins')
  yearly.columns = yearly.columns.droplevel(0)
  new_column_names = ['Extreme', 'Severe', 'Moderate', 'Mild', 'No Drought']
  yearly.columns = new_column_names
  return yearly


In [25]:
per_year_months_at_severity_level_viet = monthly_severity_per_year(spi_1m_viet)

In [26]:
per_year_months_at_severity_level_indonesia = monthly_severity_per_year(spi_1m_indonesia)

In [27]:
def categorize_spi(spi_data_array):
  """
  Categorizes SPI values into drought severity levels.

  Takes an xarray DataArray containing 12m SPI values with a 'time' coordinate.

  Returns a pandas DataFrame with the annual 'spi' and its 'category'
  """
  spi_12m_annual = spi_data_array.resample(time='1YE').last()
  spi_12m_annual_pd = spi_12m_annual.to_pandas()
  spi_12m_annual_pd.index = spi_12m_annual_pd.index.year
  spi_12m_annual_pd.index.name = 'year'
  spi_values = spi_12m_annual_pd.rename('SPI Value')

  # Define bins and labels for drought severity
  bins = [-np.inf, -2.0, -1.5, -1.0, 0, np.inf]
  labels = ['Extreme Drought', 'Severe Drought', 'Moderate Drought', 'Mild Drought', 'No Drought']

  # Categorize SPI values
  spi_category = pd.cut(spi_values, bins=bins, labels=labels, right=False)

  # Create a DataFrame from the results
  result_df = pd.DataFrame({
      'spi': spi_values,
      'category': spi_category
  })

  return result_df

In [28]:
spi_yearly_viet = categorize_spi(spi_12m_viet)
spi_yearly_indonesia = categorize_spi(spi_12m_indonesia)

In [29]:
per_year_months_at_severity_level_viet.join(spi_yearly_viet).to_csv(path_gd + 'annual_spi_vietnam.csv')

In [30]:
per_year_months_at_severity_level_indonesia.join(spi_yearly_indonesia).to_csv(path_gd + 'annual_spi_indonesia.csv')

In [31]:
vietnam = pd.read_csv(path_gd + 'annual_spi_vietnam.csv')

### Analyzing Drought Events by Year

Based on your definitions:
- A drought occurs when SPI values are less than or equal to -1.
- A drought event ends when the SPI value rises above -1.
- The severity of a drought within a given year is determined by the lowest SPI value (most negative) observed during that event *within that specific year*.
- If a drought event spans multiple years, it is counted for each year it spans.

In [32]:
drought_severity_bins = [-np.inf, -2.0, -1.5, -1.0]
drought_severity_labels = ['Extreme Drought', 'Severe Drought', 'Moderate Drought']

yearly_drought_summary = {}
current_drought_spell_data = [] # stores (time, spi_value) for current spell
in_drought = False

for time_val, spi_val in zip(spi_1m_viet.time.values, spi_1m_viet.values):
    if spi_val <= -1.0: # Drought condition
        in_drought = True
        current_drought_spell_data.append({'time': time_val, 'spi': spi_val})
    else: # Not drought condition
        if in_drought: # Drought spell just ended
            # Process the completed drought spell
            if current_drought_spell_data: # Ensure there's data in the spell
                spell_spi_values = xr.DataArray([d['spi'] for d in current_drought_spell_data],
                                                 coords={'time': [d['time'] for d in current_drought_spell_data]})

                years_in_spell = np.unique(pd.to_datetime(spell_spi_values.time.values).year)

                for year in years_in_spell:
                    if year not in yearly_drought_summary:
                        yearly_drought_summary[year] = {'event_count': 0, 'severity_counts': {'Moderate Drought': 0, 'Severe Drought': 0, 'Extreme Drought': 0}}

                    # Filter SPI values for the current year within the spell
                    spi_in_current_year = spell_spi_values.sel(time=str(year))
                    if spi_in_current_year.size > 0: # Check if there are SPI values for this year
                        max_severity_spi_for_year = spi_in_current_year.min().item() # Lowest SPI is max severity

                        # Assign to severity category
                        if max_severity_spi_for_year <= -2.0:
                            severity_label = 'Extreme Drought'
                        elif max_severity_spi_for_year <= -1.5:
                            severity_label = 'Severe Drought'
                        elif max_severity_spi_for_year <= -1.0:
                            severity_label = 'Moderate Drought'
                        else:
                            severity_label = None # Should not happen given condition spi_val <= -1.0 for a spell

                        if severity_label:
                            yearly_drought_summary[year]['event_count'] += 1
                            yearly_drought_summary[year]['severity_counts'][severity_label] += 1

            current_drought_spell_data = []
            in_drought = False

# Handle potential drought spell at the very end of the data
if in_drought and current_drought_spell_data:
    spell_spi_values = xr.DataArray([d['spi'] for d in current_drought_spell_data],
                                     coords={'time': [d['time'] for d in current_drought_spell_data]})

    years_in_spell = np.unique(pd.to_datetime(spell_spi_values.time.values).year)

    for year in years_in_spell:
        if year not in yearly_drought_summary:
            yearly_drought_summary[year] = {'event_count': 0, 'severity_counts': {'Moderate Drought': 0, 'Severe Drought': 0, 'Extreme Drought': 0}}

        spi_in_current_year = spell_spi_values.sel(time=str(year))
        if spi_in_current_year.size > 0:
            max_severity_spi_for_year = spi_in_current_year.min().item()

            if max_severity_spi_for_year <= -2.0:
                severity_label = 'Extreme Drought'
            elif max_severity_spi_for_year <= -1.5:
                severity_label = 'Severe Drought'
            elif max_severity_spi_for_year <= -1.0:
                severity_label = 'Moderate Drought'
            else:
                severity_label = None

            if severity_label:
                yearly_drought_summary[year]['event_count'] += 1
                yearly_drought_summary[year]['severity_counts'][severity_label] += 1

# Convert to a DataFrame for better display
results_df = pd.DataFrame.from_dict({
    year: {
        'Event Count': data['event_count'],
        **data['severity_counts']
    } for year, data in yearly_drought_summary.items()
}, orient='index')

results_df.index.name = 'Year'
results_df = results_df.sort_index().fillna(0).astype(int)

display(results_df)


,Event Count,Moderate Drought,Severe Drought,Extreme Drought
Year,,,,
1982,3,3,0,0
1983,2,1,1,0
1984,2,2,0,0
1985,1,0,0,1
1986,2,2,0,0
1987,4,2,2,0
1988,2,2,0,0
1989,3,1,2,0
1990,2,1,1,0
